In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os, gc
import numpy as np
import pandas as pd

OUT_DIR = "/content/drive/MyDrive/AI_Cheema_Gang/processed"

df = pd.read_parquet(os.path.join(OUT_DIR, "cicids2017_clean.parquet"))
feature_cols = pd.read_csv(os.path.join(OUT_DIR, "feature_columns.csv"))["feature"].tolist()

META_COLS = ["Flow ID","Source IP","Source Port","Destination IP",
             "Destination Port","Protocol","Timestamp"]

print(f"rows {len(df):,}   features {len(feature_cols)}")
print(f"columns: {list(df.columns[:5])} ... {list(df.columns[-4:])}")

rows 2,814,572   features 68
columns: ['Flow ID', 'Source IP', 'Source Port', 'Destination IP', 'Destination Port'] ... ['Label', 'day', 'day_order', 'label_binary']


Residual cleaning
(negative durations, sentinel values , found after Phase 2)

#Fixing leftover broken values

In [3]:
neg = df[df["Flow Duration"] < 0]

print(f"negative Flow Duration : {len(neg):,}  ({len(neg)/len(df)*100:.4f}%)")
print(f"range                  : {neg['Flow Duration'].min()} to {neg['Flow Duration'].max()}\n")

print("labels affected:")
print(neg["Label"].value_counts().to_string(), "\n")

print("as % of each class:")
tot = df["Label"].value_counts()
share = (neg["Label"].value_counts() / tot * 100).dropna().sort_values(ascending=False)
print(share.round(3).to_string(), "\n")

negcols = (df[feature_cols] < 0).sum()
print("all columns containing negatives:")
print(negcols[negcols > 0].to_string())

negative Flow Duration : 115  (0.0041%)
range                  : -13.0 to -1.0

labels affected:
Label
BENIGN    115 

as % of each class:
Label
BENIGN    0.005 

all columns containing negatives:
Flow Duration                  115
Flow Bytes/s                    85
Flow Packets/s                 115
Flow IAT Mean                  115
Flow IAT Max                   115
Flow IAT Min                  2890
Fwd IAT Min                     17
Fwd Header Length               35
Bwd Header Length               22
Init_Win_bytes_forward      995832
Init_Win_bytes_backward    1426557
min_seg_size_forward            35


In [4]:
for c in ["Init_Win_bytes_forward", "Init_Win_bytes_backward"]:
    v = df[c]
    print(f"\n{c}")
    print("  unique negative values:", sorted(v[v < 0].unique())[:5])
    print("  min positive:", v[v >= 0].min(), " max:", v.max())
    print("  protocol of negative rows:",
          df.loc[v < 0, "Protocol"].value_counts().to_dict())
    print("  protocol of positive rows:",
          df.loc[v >= 0, "Protocol"].value_counts().to_dict())

print("\nFlow IAT Min negatives:", (df["Flow IAT Min"] < 0).sum())
print(df.loc[df["Flow IAT Min"] < 0, "Label"].value_counts().head().to_dict())


Init_Win_bytes_forward
  unique negative values: [np.float64(-1.0)]
  min positive: 0.0  max: 65535.0
  protocol of negative rows: {17.0: 994197, 0.0: 1635}
  protocol of positive rows: {6.0: 1818740}

Init_Win_bytes_backward
  unique negative values: [np.float64(-1.0)]
  min positive: 0.0  max: 65535.0
  protocol of negative rows: {17.0: 994197, 6.0: 430725, 0.0: 1635}
  protocol of positive rows: {6.0: 1388015}

Flow IAT Min negatives: 2890
{'BENIGN': 2696, 'DoS Hulk': 159, 'DDoS': 19, 'DoS GoldenEye': 5, 'Heartbleed': 4}


## Step 0 — Residual Cleaning

Three data problems found after Phase 2, while inspecting the saved file
in a Parquet viewer.

Input:  cicids2017_clean.parquet  (the Phase 2 output, left untouched)
Output: cicids2017_v2.parquet

1)Fixing Negative Flow Duration
Flow Duration records how long a conversation lasted , so deleting rows with negative values

In [5]:
before = len(df)
df = df[df["Flow Duration"] >= 0].reset_index(drop=True)

print(f"removed {before - len(df)} rows with negative Flow Duration")
print(f"rows remaining: {len(df):,}")

removed 115 rows with negative Flow Duration
rows remaining: 2,814,457


2) Fixing -1 values in Init_Win_bytes

Init_Win_bytes_forward holds -1 in about 1 million rows, and
Init_Win_bytes_backward in about 1.4 million.

Checking the protocol column explains it. Every row with -1 is UDP
traffic, or a flow where the other side never replied. "Window bytes" is
a TCP-only measurement, so the tool writes -1 to mean "does not apply
here".

This matters because the real values range from 0 to 65535. If we leave
-1 in place, the model reads it as "an extremely small window" instead of
"no window exists". So we do two things:

  1. Replace -1 with 0
  2. Add a yes/no column marking that the value was not applicable

In [6]:
for c in ["Init_Win_bytes_forward", "Init_Win_bytes_backward"]:
    flag = c + "_missing"
    df[flag] = (df[c] < 0).astype("int8")
    df.loc[df[c] < 0, c] = 0
    print(f"{c}: {df[flag].sum():,} values marked not-applicable and set to 0")

feature_cols = feature_cols + ["Init_Win_bytes_forward_missing",
                               "Init_Win_bytes_backward_missing"]

print(f"\nfeatures now: {len(feature_cols)}")

Init_Win_bytes_forward: 995,832 values marked not-applicable and set to 0
Init_Win_bytes_backward: 1,426,557 values marked not-applicable and set to 0

features now: 70


Fixing Flow IAT Min negative values

FLow IAT contains 2,890 negative
values but the issue is, Four of these rows are Heartbleed, and
Heartbleed has only 11 rows in the whole dataset

So we set the negative values to 0 and keep every row.

In [7]:
negcols = [c for c in feature_cols if (df[c] < 0).any()]
clamped = int(sum((df[c] < 0).sum() for c in negcols))

for c in negcols:
    df.loc[df[c] < 0, c] = 0

print(f"set {clamped:,} negative values to 0 across {len(negcols)} columns:")
for c in negcols:
    print("  ", c)
print(f"\nnegatives remaining: {int((df[feature_cols] < 0).sum().sum())}")

set 2,884 negative values to 0 across 5 columns:
   Flow IAT Min
   Fwd IAT Min
   Fwd Header Length
   Bwd Header Length
   min_seg_size_forward

negatives remaining: 0


Save a parquet file with new version "cicids2017_v2.parquet"

In [8]:
V2 = os.path.join(OUT_DIR, "cicids2017_v2.parquet")
df.to_parquet(V2, index=False)

pd.Series(feature_cols).to_csv(os.path.join(OUT_DIR, "feature_columns_v2.csv"),
                               index=False, header=["feature"])

print(f"saved -> {V2}")
print(f"rows {len(df):,}   features {len(feature_cols)}")
print(f"\nclass counts unchanged?")
print(df['Label'].value_counts().to_string())

saved -> /content/drive/MyDrive/AI_Cheema_Gang/processed/cicids2017_v2.parquet
rows 2,814,457   features 70

class counts unchanged?
Label
BENIGN                        2265489
DoS Hulk                       222563
PortScan                       158804
DDoS                           128025
DoS GoldenEye                   10293
FTP-Patator                      7935
SSH-Patator                      5897
DoS slowloris                    5769
DoS Slowhttptest                 5499
Bot                              1956
Web Attack - Brute Force         1507
Web Attack - XSS                  652
Infiltration                       36
Web Attack - Sql Injection         21
Heartbleed                         11


## Step 1 — Train / Test Split

We split inside each day: the earliest 70% of each day's flows go to
training, the latest 30% go to testing.

#Random split
A random split scatters near identical flows across both sides.
Attack traffic comes in bursts of very similar connections, so the model would
effectively see test rows during training and score far higher than it
deserves. This is the main reason published CICIDS2017 results often
report ~99.8% accuracy.

# Why not split by whole days

Our first plan was Monday-Wednesday for training and Thursday-Friday for
testing. Checking the attack schedule showed this does not work:

   Train days would contain: brute force, DoS, Heartbleed
   Test days would contain:  web attacks, infiltration, botnet, port scan, DDoS

Not a single attack type appears on both sides. A model cannot predict a
class it has never seen, so multiclass classification would be impossible
and every test attack would be wrong by construction

# Why within-day works

Attacks ran for sustained periods, so cutting each day at 70% leaves
examples of the same attack on both sides.

# What this split does NOT give us

CICIDS2017 contains one execution of each attack type. Splitting it means
the DoS Hulk in training and the DoS Hulk in testing are the same attack
run, same tool, same machines. So our results measure whether a model can
keep detecting an attack it is currently watching. They do not measure
generalisation to genuinely new attacks. No split of this dataset can.

In [12]:
GAP = pd.Timedelta(minutes=1)
TRAIN_FRAC = 0.70

df = df.sort_values(["day_order", "Timestamp"]).reset_index(drop=True)

parts = []
for day, g in df.groupby("day", sort=False):
    g = g.sort_values("Timestamp")
    cut_time = g["Timestamp"].iloc[int(len(g) * TRAIN_FRAC)]

    g = g.copy()
    g["split"] = np.where(g["Timestamp"] < cut_time - GAP, "train",
                 np.where(g["Timestamp"] > cut_time + GAP, "test", "gap"))

    n_gap = (g["split"] == "gap").sum()
    print(f"{day:<10} cut at {cut_time.strftime('%H:%M:%S')}   "
          f"train {(g['split']=='train').sum():>7,}   "
          f"test {(g['split']=='test').sum():>7,}   "
          f"dropped {n_gap:,}")
    parts.append(g)

df = pd.concat(parts).sort_values(["day_order", "Timestamp"]).reset_index(drop=True)

dropped = (df["split"] == "gap").sum()
df = df[df["split"] != "gap"].reset_index(drop=True)

print(f"\ndropped {dropped:,} rows in boundary gaps ({dropped/2814457*100:.3f}%)")
print(f"train {(df.split=='train').sum():,}   test {(df.split=='test').sum():,}")

Monday     cut at 14:05:50   train 368,767   test 158,206   dropped 1,510
Tuesday    cut at 14:39:00   train 308,656   test 130,613   dropped 5,079
Wednesday  cut at 13:27:00   train 475,074   test 203,970   dropped 3,296
Thursday   cut at 15:19:00   train 317,242   test 128,644   dropped 11,551
Friday     cut at 15:48:00   train 491,023   test 210,079   dropped 747

dropped 22,183 rows in boundary gaps (0.788%)
train 1,960,762   test 831,512


In [13]:
tab = pd.crosstab(df["Label"], df["split"])[["train", "test"]]
tab["total"] = tab.sum(axis=1)
print(tab.sort_values("total", ascending=False).to_string(), "\n")

missing_test  = tab[tab["test"] == 0].index.tolist()
missing_train = tab[tab["train"] == 0].index.tolist()
too_few       = tab[(tab["test"] > 0) & (tab["test"] < 30)].index.tolist()

print("classes absent from TEST :", missing_test or "none")
print("classes absent from TRAIN:", missing_train or "none")
print("classes with <30 test rows (not statistically meaningful):", too_few or "none")

split                         train    test    total
Label                                               
BENIGN                      1542990  700627  2243617
DoS Hulk                     222563       0   222563
PortScan                     158804       0   158804
DDoS                              0  128025   128025
DoS GoldenEye                 10293       0    10293
FTP-Patator                    7935       0     7935
DoS slowloris                  5764       5     5769
SSH-Patator                    2754    2833     5587
DoS Slowhttptest               5499       0     5499
Bot                            1956       0     1956
Web Attack - Brute Force       1507       0     1507
Web Attack - XSS                652       0      652
Infiltration                     24      11       35
Web Attack - Sql Injection       21       0       21
Heartbleed                        0      11       11 

classes absent from TEST : ['Bot', 'DoS GoldenEye', 'DoS Hulk', 'DoS Slowhttptest', 'FTP-Patator'

# our this option for split failed , Trying Split based upon each class ratio 30-70

In [9]:
TRAIN_FRAC = 0.70

parts = []
for lab, g in df.groupby("Label", sort=False):
    g = g.sort_values("Timestamp").copy()
    k = max(1, int(len(g) * TRAIN_FRAC))
    g["split"] = ["train"] * k + ["test"] * (len(g) - k)
    parts.append(g)

df = pd.concat(parts).sort_values(["day_order", "Timestamp"]).reset_index(drop=True)

n_tr = (df["split"] == "train").sum()
n_te = (df["split"] == "test").sum()
print(f"train {n_tr:,} ({n_tr/len(df)*100:.1f}%)")
print(f"test  {n_te:,} ({n_te/len(df)*100:.1f}%)")

train 1,970,113 (70.0%)
test  844,344 (30.0%)


In [10]:
tab = pd.crosstab(df["Label"], df["split"])[["train", "test"]]
tab["total"] = tab.sum(axis=1)
print(tab.sort_values("total", ascending=False).to_string())

print("\nabsent from test :", tab[tab["test"] == 0].index.tolist() or "none")
print("absent from train:", tab[tab["train"] == 0].index.tolist() or "none")
print("fewer than 30 test rows:", tab[(tab.test > 0) & (tab.test < 30)].index.tolist() or "none")

split                         train    test    total
Label                                               
BENIGN                      1585842  679647  2265489
DoS Hulk                     155794   66769   222563
PortScan                     111162   47642   158804
DDoS                          89617   38408   128025
DoS GoldenEye                  7205    3088    10293
FTP-Patator                    5554    2381     7935
SSH-Patator                    4127    1770     5897
DoS slowloris                  4038    1731     5769
DoS Slowhttptest               3849    1650     5499
Bot                            1369     587     1956
Web Attack - Brute Force       1054     453     1507
Web Attack - XSS                456     196      652
Infiltration                     25      11       36
Web Attack - Sql Injection       14       7       21
Heartbleed                        7       4       11

absent from test : none
absent from train: none
fewer than 30 test rows: ['Heartbleed', 'Infiltrat

In [11]:
bad = []
for lab, g in df.groupby("Label"):
    last_train = g.loc[g.split == "train", "Timestamp"].max()
    first_test = g.loc[g.split == "test", "Timestamp"].min()
    if pd.notna(last_train) and pd.notna(first_test) and first_test < last_train:
        bad.append(lab)
print("classes where a test flow precedes a training flow:", bad or "none")

classes where a test flow precedes a training flow: none


# Step 2 — Feature Scaling

70 features are on wildly different scale. Later NN would treat  attributes with bigger number more importance

In [12]:
from sklearn.preprocessing import StandardScaler
import joblib

flag_cols  = [c for c in feature_cols if c.endswith("_missing")]
numer_cols = [c for c in feature_cols if c not in flag_cols]

print(f"{len(numer_cols)} numeric features, {len(flag_cols)} binary flags")

# 1. log1p on the skewed numeric features
df[numer_cols] = np.log1p(df[numer_cols].clip(lower=0))

# 2. fit the scaler on TRAIN ONLY
train_mask = df["split"] == "train"
scaler = StandardScaler()
scaler.fit(df.loc[train_mask, numer_cols])

df[numer_cols] = scaler.transform(df[numer_cols]).astype("float32")

joblib.dump(scaler, os.path.join(OUT_DIR, "scaler.joblib"))
print("scaler fitted on", train_mask.sum(), "training rows, saved to Drive")

68 numeric features, 2 binary flags
scaler fitted on 1970113 training rows, saved to Drive


Verification

In [13]:
tr = df.loc[df["split"] == "train", numer_cols]
te = df.loc[df["split"] == "test",  numer_cols]

print("TRAIN   mean %.4f   std %.4f" % (tr.values.mean(), tr.values.std()))
print("TEST    mean %.4f   std %.4f" % (te.values.mean(), te.values.std()))
print("\nany NaN or inf:", df[feature_cols].isna().sum().sum(),
      int(np.isinf(df[feature_cols].values).sum()))
print("value range: %.2f to %.2f" % (df[feature_cols].values.min(),
                                      df[feature_cols].values.max()))

TRAIN   mean -0.0000   std 0.9852
TEST    mean -0.0328   std 0.9747

any NaN or inf: 0 0
value range: -12.99 to 61.78


# Step 3 — Label Encoding

Two targets come out of this:

  label_binary  0 = normal, 1 = attack        (created in Phase 2)
  label_multi   0 to 14, one per class        (created here)

In [14]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df["label_multi"] = le.fit_transform(df["Label"])

mapping = dict(zip(le.classes_, range(len(le.classes_))))

print("code  class")
print("-" * 40)
for name, code in mapping.items():
    print(f"{code:>4}  {name}")

pd.Series(mapping).to_csv(os.path.join(OUT_DIR, "label_mapping.csv"),
                          header=["code"])
print(f"\nmapping saved to label_mapping.csv")

code  class
----------------------------------------
   0  BENIGN
   1  Bot
   2  DDoS
   3  DoS GoldenEye
   4  DoS Hulk
   5  DoS Slowhttptest
   6  DoS slowloris
   7  FTP-Patator
   8  Heartbleed
   9  Infiltration
  10  PortScan
  11  SSH-Patator
  12  Web Attack - Brute Force
  13  Web Attack - Sql Injection
  14  Web Attack - XSS

mapping saved to label_mapping.csv


In [15]:
print("label_multi range :", df["label_multi"].min(), "to", df["label_multi"].max())
print("unique codes      :", df["label_multi"].nunique())
print("any nulls         :", df["label_multi"].isna().sum())

print("\ncheck the two targets agree:")
chk = df.groupby("Label")[["label_multi", "label_binary"]].first()
chk["count"] = df["Label"].value_counts()
print(chk.to_string())

label_multi range : 0 to 14
unique codes      : 15
any nulls         : 0

check the two targets agree:
                            label_multi  label_binary    count
Label                                                         
BENIGN                                0             0  2265489
Bot                                   1             1     1956
DDoS                                  2             1   128025
DoS GoldenEye                         3             1    10293
DoS Hulk                              4             1   222563
DoS Slowhttptest                      5             1     5499
DoS slowloris                         6             1     5769
FTP-Patator                           7             1     7935
Heartbleed                            8             1       11
Infiltration                          9             1       36
PortScan                             10             1   158804
SSH-Patator                          11             1     5897
Web Attack - Br

## Step 4 — Building the Graph

#Node :
endpoint - one node per machine+port (IP:port)
host - one node per machine (IP)

Each flow becomes an edge. The 70 scaled features ride on that edge

time windows:One graph with 2.8 million edges will not fit in memory. We slice the
traffic into fixed time windows and build one small graph per window.

In [16]:
def node_ids(frame, mode):
    if mode == "endpoint":
        s = frame["Source IP"] + ":" + frame["Source Port"].astype(int).astype(str)
        d = frame["Destination IP"] + ":" + frame["Destination Port"].astype(int).astype(str)
    else:
        s, d = frame["Source IP"].copy(), frame["Destination IP"].copy()
    return s, d


print(f"{'class':<16}{'mode':<10}{'nodes':>10}{'uniq edges':>12}{'max nbrs':>10}")
print("-" * 58)
for lab in ["PortScan", "DoS Hulk", "DDoS", "Bot", "BENIGN"]:
    sub = df[df["Label"] == lab]
    for mode in ["host", "endpoint"]:
        s, d = node_ids(sub, mode)
        pairs = pd.DataFrame({"s": s, "d": d}).drop_duplicates()
        print(f"{lab if mode=='host' else '':<16}{mode:<10}"
              f"{pd.concat([s,d]).nunique():>10,}"
              f"{pairs.shape[0]:>12,}"
              f"{pairs.groupby('s').size().max():>10,}")
    print()

class           mode           nodes  uniq edges  max nbrs
----------------------------------------------------------
PortScan        host               2           1         1
                endpoint      15,142     158,550       995

DoS Hulk        host               2           1         1
                endpoint      14,109      14,108         1

DDoS            host               2           2         1
                endpoint      45,393      45,395         3

Bot             host              10          12         5
                endpoint       1,231       1,929       701

BENIGN          host          19,129     113,743     6,657
                endpoint     420,752   1,240,815    23,897



Postscan 2 unique nodes in Host  
while 15K nodes and 150 K edges and a maximum degree number this shows that

The attacker’s whole method is changing the port every time. If you ignore the port, every flow looks identical. If you keep it, every flow is different.

In [17]:
NODE_MODE   = "endpoint"      # set from the numbers above
WINDOW      = "5min"

src, dst = node_ids(df, NODE_MODE)

all_nodes = pd.Index(pd.concat([src, dst]).unique())
node_map  = pd.Series(range(len(all_nodes)), index=all_nodes)

df["src_id"] = src.map(node_map).astype("int32")
df["dst_id"] = dst.map(node_map).astype("int32")

print(f"mode: {NODE_MODE}")
print(f"total nodes : {len(all_nodes):,}")
print(f"total edges : {len(df):,}")
print(f"avg degree  : {2*len(df)/len(all_nodes):.1f}")

mode: endpoint
total nodes : 445,119
total edges : 2,814,457
avg degree  : 12.6


In [18]:
for lab in ["PortScan", "DoS Hulk", "DDoS", "Bot"]:
    sub = df[df["Label"] == lab]
    s, d = node_ids(sub, "endpoint")
    pairs = pd.DataFrame({"s": s, "d": d}).drop_duplicates()
    print(f"{lab:<12} out: {pairs.groupby('s').size().max():>7,}   "
          f"in: {pairs.groupby('d').size().max():>7,}")

PortScan     out:     995   in:     312
DoS Hulk     out:       1   in:  14,108
DDoS         out:       3   in:  45,392
Bot          out:     701   in:   1,226


out = the busiest sender talks to this many different places
in = the busiest receiver gets traffic from this many different places

all four attacks have strong graph structure.

Step 5--Build the graph tensors

In [20]:
!pip install -q torch_geometric
import torch
from torch_geometric.data import Data
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())

torch 2.11.0+cpu | cuda: False


In [21]:
import pickle

WINDOW = "5min"
df["window"] = df["Timestamp"].dt.floor(WINDOW)

GRAPH_DIR = os.path.join(OUT_DIR, "graphs")
os.makedirs(GRAPH_DIR, exist_ok=True)

def build_window(g):
    """One time window -> one PyG graph. Nodes renumbered 0..N-1 locally."""
    nodes = pd.Index(pd.concat([g["src_id"], g["dst_id"]]).unique())
    local = pd.Series(range(len(nodes)), index=nodes)

    edge_index = torch.tensor(
        np.vstack([g["src_id"].map(local).values,
                   g["dst_id"].map(local).values]), dtype=torch.long)

    return Data(
        edge_index = edge_index,
        edge_attr  = torch.tensor(g[feature_cols].values, dtype=torch.float),
        y          = torch.tensor(g["label_multi"].values, dtype=torch.long),
        y_bin      = torch.tensor(g["label_binary"].values, dtype=torch.long),
        num_nodes  = len(nodes),
    )

for split in ["train", "test"]:
    graphs = []
    sub = df[df["split"] == split]
    for win, g in sub.groupby("window", sort=True):
        if len(g) < 2:
            continue
        graphs.append(build_window(g))

    path = os.path.join(GRAPH_DIR, f"{split}_{WINDOW}.pkl")
    with open(path, "wb") as f:
        pickle.dump(graphs, f)

    edges = sum(gr.edge_index.shape[1] for gr in graphs)
    nodes = sum(gr.num_nodes for gr in graphs)
    print(f"{split:<6} {len(graphs):>4} graphs   {nodes:>9,} nodes   {edges:>9,} edges")
    print(f"       saved -> {path}")
    del graphs; gc.collect()

train   383 graphs   1,407,785 nodes   1,970,107 edges
       saved -> /content/drive/MyDrive/AI_Cheema_Gang/processed/graphs/train_5min.pkl
test    173 graphs     622,277 nodes     844,342 edges
       saved -> /content/drive/MyDrive/AI_Cheema_Gang/processed/graphs/test_5min.pkl


Turned the table into pictures that a graph model can read.
We cut the whole week dataset into 5-minute slices
Before One big spreadsheet, 2.79 million rows.
After 496 small graphs, one per 5-minute slice of the week.

496 pictures — the week is about 41 hours of captured traffic, and 41 hours ÷ 5 minutes ≈ 496.

Globally your dots are numbered 0 to 444,024. But one 5-minute chunk only uses 1,388 of them.

Verify

In [22]:
with open(os.path.join(GRAPH_DIR, f"train_{WINDOW}.pkl"), "rb") as f:
    gs = pickle.load(f)

g = gs[len(gs)//2]
print(g)
print(f"\nnodes      : {g.num_nodes:,}")
print(f"edges      : {g.edge_index.shape[1]:,}")
print(f"features   : {g.edge_attr.shape}")
print(f"attacks    : {int(g.y_bin.sum())} of {len(g.y_bin)}")
print(f"node ids   : {g.edge_index.min().item()} to {g.edge_index.max().item()}")

Data(edge_index=[2, 7774], edge_attr=[7774, 70], y=[7774], y_bin=[7774], num_nodes=6366)

nodes      : 6,366
edges      : 7,774
features   : torch.Size([7774, 70])
attacks    : 0 of 7774
node ids   : 0 to 6365


Verified a single grapgh
It has 1,692 lines, each carrying 70 columns/features

In [23]:
df.to_parquet(V2, index=False)

print(f"saved {len(df):,} rows, {df.shape[1]} columns")
print("columns:", [c for c in df.columns if c not in feature_cols])
print("\nsanity check — features should be scaled (mean near 0):")
print("  train mean %.4f" % df.loc[df.split=="train", feature_cols].values.mean())
print("  label_multi present:", "label_multi" in df.columns)
print("  split present      :", "split" in df.columns)

saved 2,814,457 rows, 86 columns
columns: ['Flow ID', 'Source IP', 'Source Port', 'Destination IP', 'Destination Port', 'Protocol', 'Timestamp', 'Label', 'day', 'day_order', 'label_binary', 'split', 'label_multi', 'src_id', 'dst_id', 'window']

sanity check — features should be scaled (mean near 0):
  train mean 0.0124
  label_multi present: True
  split present      : True
